# Solution in the adiabatic limit : $\dot B = \dot P_B =0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g

A_all   = np.arange(-5.2, 5.2, 0.01)
PA_all  = np.arange(-5.2, 5.2, 0.01)
A, PA   = np.meshgrid(A_all, PA_all)

A_0   = np.sqrt(eps_d / g)
delta = kappa_a * kappa_b / 4 / g**2

H_vals = (-PA**2 * kappa_a / 2
          + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * A**2 * PA**2)
          * 2 * g**2 / kappa_b)

plt.figure()
plt.pcolormesh(A_all, PA_all, H_vals, cmap='RdBu_r')
plt.colorbar(label='Hamiltonian')
cs = plt.contour(A_all, PA_all, H_vals, levels=[0.0], colors='white', linewidths=2)
plt.scatter(np.sqrt(A_0**2 - delta), 0, color='black', s=10)
plt.xlabel('A')
plt.ylabel(r'$P_A$')
plt.tight_layout()
plt.show()

In [ ]:
paths = cs.get_paths()
contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

A_contour  = contour_coords[:, 0]
PA_contour = contour_coords[:, 1]

begin = 1690#2090
end = 2685#3225

A_keep = A_contour[begin:end]
PA_keep = PA_contour[begin:end]

A_0_p = np.sqrt(A_0**2-delta)
A_keep = np.insert(A_keep,0,A_0_p)
A_keep = np.append(A_keep,0)

PA_keep = np.insert(PA_keep,0,0)
PA_keep = np.append(PA_keep,0)

PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g

fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
ax_1.set_xlabel(r'$A$')
ax_1.set_ylabel(r'$P_A$')

sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
ax_2.set_xlabel(r'$B$')
ax_2.set_ylabel(r'$P_B$')

plt.colorbar(sc, ax=ax_2)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.interpolate import interp1d

def reparametrize(string, new_size=None):
    # arc-length parametrization
    ds = np.linalg.norm(np.diff(string, axis=0), axis=1)
    if new_size is None:
        new_size = len(string)

    s = np.zeros(len(string))   # must match len(string), not new_size
    s[1:] = np.cumsum(ds)

    if s[-1] == 0:
        return string.copy()

    s_uniform = np.linspace(0, s[-1], new_size)  # new_size points for output

    A_interp  = interp1d(s, string[:, 0], kind='linear')
    PA_interp = interp1d(s, string[:, 1], kind='linear')
    B_interp  = interp1d(s, string[:, 2], kind='linear')
    PB_interp = interp1d(s, string[:, 3], kind='linear')

    new_string = np.zeros((new_size, string.shape[1]))
    new_string[:, 0] = A_interp(s_uniform)
    new_string[:, 1] = PA_interp(s_uniform)
    new_string[:, 2] = B_interp(s_uniform)
    new_string[:, 3] = PB_interp(s_uniform)
    return new_string

In [ ]:
string = np.stack([A_keep, PA_keep, B_keep, PB_keep], axis=1)
N=100
def initialize_string(N):
    string_new = reparametrize(string, new_size=N)
    return(string_new)
string_init = initialize_string(N)
A_new = string_init[:,0]
PA_new = string_init[:,1]
B_new = string_init[:,2]
PB_new = string_init[:,3]

In [ ]:
fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.scatter(A_new,PA_new, c=range(len(A_new)))
ax_1.set_xlabel(r'$A$')
ax_1.set_ylabel(r'$P_A$')

sc = ax_2.scatter(B_new, PB_new, c=range(len(A_new)))
ax_2.set_xlabel(r'$B$')
ax_2.set_ylabel(r'$P_B$')

plt.colorbar(sc, ax=ax_2)
plt.tight_layout()
plt.show()

# Full path in 4D beyond the adiabatic limit

In [ ]:
c=np.sqrt(2)*g
def F_loss_a(X):
    A, PA, B, PB = X
    dA = c*(A*B - PA*PB) + kappa_a * (A/2+PA)
    dPA =   c * (A * PB - PA * B) - kappa_a * PA/2
    dB =  eps_d/np.sqrt(2) - g*(A**2 + PA**2)/np.sqrt(2) + kappa_b * (B/2 + PB)
    dPB = - c * A * PA - kappa_b * PB/2
    return np.array([dA, dPA, dB, dPB])

def tangent(string):
    T = np.zeros_like(string)
    T[1:-1] = string[2:] - string[:-2]
    T[0] = string[1] - string[0]
    T[-1] = string[-1] - string[-2]
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return T / norms

def jacobian(X):
    A, PA, B, PB = X

    J = np.array([
        [c*B+kappa_a/2, -c*PB+kappa_a, c*A, -c*PA],
        [c*PB, -c*B-kappa_a/2, -c*PA, c*A],
        [-c*A, -c*PA, kappa_b/2, kappa_b],
        [-c*PA, -c*A, 0, -kappa_b/2]             
    ])

    return J

def semi_implicit_normal_update(xi, Ti, F, dtau, reg=0.0):
    """
    One semi-implicit update for a single string point xi with unit
    tangent Ti. Solves
 
        (I - dtau * J_perp) dx = dtau * F_perp(xi)
 
    instead of the explicit dx = dtau * F_perp(xi), where J_perp is F's
    local Jacobian at xi restricted to the subspace normal to Ti.
    Returns dx (which lies purely in that normal subspace, up to
    floating-point error).
 
    reg: optional small Tikhonov term added to the solve, as a safety
    valve if (I - dtau*J_perp) is ever near-singular.
    """
    xi = np.asarray(xi, dtype=float)
    n = xi.size
    I = np.eye(n)
 
    # Projector onto the subspace normal to the local tangent.
    P_perp = I - np.outer(Ti, Ti)
 
    Fi = np.asarray(F(xi), dtype=float)
    Fi_normal = P_perp @ Fi
 
    # Local Jacobian of F at xi, restricted to the normal subspace.
    Ji = jacobian(xi)
    J_perp = P_perp @ Ji @ P_perp
 
    A_mat = I - dtau * J_perp
    if reg > 0.0:
        A_mat = A_mat + reg * I
    rhs = dtau * Fi_normal
 
    dx = np.linalg.solve(A_mat, rhs)
 
    # dx should already lie purely in the normal subspace: because
    # J_perp @ Ti = 0 exactly, the tangential component of the solve
    # vanishes analytically. Reproject to strip floating-point leakage.
    return P_perp @ dx
 
 
def string_step(string, F, dtau, init_point, end_point, reg=0.0):
    """
    Semi-implicit string-method step.
 
    Same interface as the original explicit version, but each interior
    point is advanced with an implicit solve against the local transverse
    Jacobian instead of a plain dtau*F_perp kick. This is the fix
    recommended in 05_numerics_and_open_questions.md: it removes the
    exponential blow-up seen with explicit stepping, because the update
    multiplies the transverse error by 1/(1 - dtau*lambda) instead of
    (1 + dtau*lambda) for the local transverse eigenvalue lambda — bounded
    (and in fact damping, for large enough dtau) regardless of the sign of
    lambda, unlike the explicit multiplier which is unbounded for either
    sign as dtau or lambda grow.
    """
    new_string = string.copy()
    T = tangent(string)
 
    for i in range(1, len(string) - 1):
        dx = semi_implicit_normal_update(
            string[i], T[i], F, dtau, reg=reg
        )
        new_string[i] = string[i] + dx
 
    # fix endpoints
    new_string[0] = init_point
    new_string[-1] = end_point
 
    # redistribute images
    new_string = reparametrize(new_string)
 
    # fix endpoints again after interpolation
    new_string[0] = init_point
    new_string[-1] = end_point
 
    return new_string

def residual_norm(string):
    T = tangent(string)
    vals = []
    for i in range(1, len(string)-1):
        Fi = F_loss_a(string[i])
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]
        vals.append(np.linalg.norm(Fi_normal))
    return max(vals) if vals else 0.0

def hamiltonian(x):
    A,PA,B,PB= x
    return -eps_d*PB/np.sqrt(2)-np.sqrt(2)*g*A*PA*B+g*(A**2+PA**2)/np.sqrt(2)-kappa_a*(A+PA)*PA/2-kappa_b*(B+PB)*PB/2

def hamiltonian_string(string):
    h_vals = []
    for x in string:
        h_vals.append(hamiltonian(x))
    return max(h_vals) if h_vals else 0.0

def string_method(initialize_string, F, N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    string = initialize_string(N)
    init_point = string[0]
    end_point = string[-1]
    res_list = []
    h_disc_list = []
    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, F, dtau, init_point, end_point)

        err = np.max(np.linalg.norm(string - old, axis=1))
        res = residual_norm(string)
        res_list.append(res)
        h_disc = hamiltonian_string(string)
        h_disc_list.append(h_disc)
        if k % 200 == 0:
            print(f"iter={k}, err={err:.3e}, res={res:.3e}, h_disc={h_disc:.3e}")

        if err < tol:
            print(f"converged at iter={k}, err={err:.3e}")
            break
    return string, res_list, h_disc

In [ ]:
string_test_loss, res_list, h_disc = string_method(initialize_string, F_loss_a, dtau=1e-3, max_iter = 5_000)

In [ ]:
fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.plot(string_init[:,0],string_init[:,1], label="init")
ax_1.plot(string_test_loss[:,0],string_test_loss[:,1])
ax_1.set_xlabel(r'$A$')
ax_1.set_ylabel(r'$P_A$')
ax_1.legend()

ax_2.plot(string_init[:,2],string_init[:,3], label="init")
ax_2.plot(string_test_loss[:,2],string_test_loss[:,3])
ax_2.set_xlabel(r'$B$')
ax_2.set_ylabel(r'$P_B$')
ax_2.legend()

plt.tight_layout()
plt.show()